In [ ]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("retail_data.db")

# Advanced SQL Query using CTEs and Window Functions
query = """
WITH DailySales AS (
    -- Step 1: Aggregate daily revenue and quantity sold
    SELECT 
        order_date,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(quantity) AS total_units_sold,
        ROUND(SUM(quantity * unit_price), 2) AS daily_revenue
    FROM transactions
    GROUP BY order_date
),
SalesMetrics AS (
    -- Step 2: Apply Window Functions for 7-day Moving Average & Previous Day Revenue
    SELECT 
        order_date,
        daily_revenue,
        -- 7-Day Moving Average of Revenue
        ROUND(AVG(daily_revenue) OVER (
            ORDER BY order_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_avg_7d,
        -- Lag function to get previous day's revenue
        LAG(daily_revenue, 1) OVER (ORDER BY order_date) AS prev_day_revenue
    FROM DailySales
)
-- Step 3: Final Selection with Day-over-Day Growth (%)
SELECT 
    order_date,
    daily_revenue,
    rolling_avg_7d,
    prev_day_revenue,
    ROUND(((daily_revenue - prev_day_revenue) / prev_day_revenue) * 100, 2) AS dod_growth_pct
FROM SalesMetrics
ORDER BY order_date;
"""

# Execute query and load into Pandas DataFrame
df_metrics = pd.read_sql_query(query, conn)
conn.close()

# Display the first 10 rows
df_metrics.head(10)
